In [ ]:
import csv
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn, optim
from torchvision import datasets, transforms

sys.path.append(os.path.abspath(".."))
from src.architectures import GeneralMLP
from src.continual_learning import GPM
from src.utils import apply_heavy_tailed_init, set_seed

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- 1. GPU-Accelerated Data Loading ---
def generate_permutations(num_tasks, num_pixels=784, seed=42):
    """Generates clean pseudorandom domain permutations mapping each task stream."""
    rng = np.random.RandomState(seed)
    perms = [
        torch.arange(num_pixels).to(DEVICE)
    ]  # Task 0 is standard un-permuted MNIST
    for _ in range(num_tasks - 1):
        perms.append(torch.from_numpy(rng.permutation(num_pixels)).to(DEVICE))
    return perms


def get_gpu_data(dataset):
    """Loads raw tensors to GPU once to avoid repetitive overhead."""
    imgs = torch.stack([img for img, _ in dataset]).to(DEVICE).view(-1, 784)
    lbls = torch.tensor([lbl for _, lbl in dataset]).to(DEVICE)
    return imgs, lbls


def save_physics_snapshot(model, input_batch, output_dir, t_idx, epoch, alpha, g):
    model.eval()

    # Capture pre-activations
    pre_acts = model.get_pre_activations(input_batch)
    linear_layers = [m for m in model.modules() if isinstance(m, nn.Linear)]

    layer_physics = {}
    for idx, layer in enumerate(linear_layers):
        layer_key = f"layer_{idx}" if idx < len(linear_layers) - 1 else "classifier"
        # Extract pre-activation for this specific layer
        h = pre_acts[idx] if idx < len(linear_layers) - 1 else pre_acts["classifier"]

        layer_physics[layer_key] = {"pre_activations": h.float().cpu()}

    snapshot = {
        "metadata": {"task": t_idx + 1, "epoch": epoch, "alpha": alpha, "g": g},
        "state_dict": model.state_dict(),  # Weights W are stored here
        "physics_data": layer_physics,  # Only storing h to save space
    }

    output_dir.mkdir(parents=True, exist_ok=True)
    file_path = output_dir / f"snapshot_T{t_idx + 1}_E{epoch}.pt"

    # Using weights + pre_acts allows reconstruction of J, Rank, and CKA later
    torch.save(snapshot, file_path)
    return file_path


class CLMetricsTracker:
    def __init__(self, max_tasks=20):
        self.max_tasks = max_tasks
        # defaultdict-style dynamic storage to avoid rigid pre-allocation
        self.history = {}
        self._total_steps_logged = 0

    def _ensure_key_exists(self, key):
        """Lazy initialization for new dynamic metric keys."""
        if key not in self.history:
            # Pad retroactively with None for previous steps if a metric is added late
            self.history[key] = [None] * self._total_steps_logged

    def log(self, step, acc_list, **extra_metrics):
        """
        Logs a single evaluation step.

        Args:
            step: Global step index.
            acc_list: List of task accuracies [acc_t0, acc_t1, ...]
            **extra_metrics: Arbitrary method-specific key-value pairs.
                             Examples:
                               basis_rank=[12, 18, 5]
                               cum_rank=35
                               effective_rank={"layer1": 4.2, "layer2": 8.1}
        """
        self._ensure_key_exists("step")
        self.history["step"].append(step)

        # 1. Log Task Accuracies
        for t_idx in range(self.max_tasks):
            col_key = f"task_{t_idx}_acc"
            self._ensure_key_exists(col_key)
            if t_idx < len(acc_list):
                self.history[col_key].append(float(acc_list[t_idx]))
            else:
                self.history[col_key].append(None)

        # 2. Dynamically Log Extra Method-Specific Metrics
        for metric_name, val in extra_metrics.items():
            if isinstance(val, (list, tuple)):
                # Handle sequence inputs (e.g., basis_rank per task or per layer)
                for idx, sub_val in enumerate(val):
                    sub_key = f"{metric_name}_{idx}"
                    self._ensure_key_exists(sub_key)
                    self.history[sub_key].append(
                        None if sub_val is None else float(sub_val)
                    )
            elif isinstance(val, dict):
                # Handle dictionary inputs (e.g., {"layer1": 4.5, "layer2": 8.2})
                for dict_key, sub_val in val.items():
                    sub_key = f"{metric_name}_{dict_key}"
                    self._ensure_key_exists(sub_key)
                    self.history[sub_key].append(
                        None if sub_val is None else float(sub_val)
                    )
            else:
                # Handle single scalar values
                self._ensure_key_exists(metric_name)
                self.history[metric_name].append(None if val is None else float(val))

        # 3. Pad any keys that were created previously but NOT passed in this log call
        self._total_steps_logged += 1
        for key in self.history:
            if len(self.history[key]) < self._total_steps_logged:
                self.history[key].append(None)

    def save_to_csv(self, filepath="cl_experiment_metrics.csv"):
        """Exports all recorded columns into a clean, flat CSV file."""
        directory = os.path.dirname(filepath)
        if directory and not os.path.exists(directory):
            os.makedirs(directory)

        # Gather headers dynamically, prioritizing 'step' first
        headers = ["step"] + [k for k in self.history if k != "step"]

        # Drop columns that are completely empty (all None)
        active_headers = [
            h for h in headers if any(val is not None for val in self.history[h])
        ]

        num_rows = self._total_steps_logged

        with open(filepath, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(active_headers)

            for r_idx in range(num_rows):
                row_data = [
                    "" if self.history[h][r_idx] is None else self.history[h][r_idx]
                    for h in active_headers
                ]
                writer.writerow(row_data)

        print(f"Metrics successfully exported to: '{filepath}'")

In [ ]:
import os
from pathlib import Path

import torch

# --- 1. GLOBAL EXPERIMENT HYPERPARAMETERS ---
RUN_SEEDS = [2, 3, 4]
NUM_TASKS = 20
EPOCHS_PER_TASK = 5
LR_TASK_0 = 1e-2  # Higher LR to rapidly align the shared backbone
LR_SUBSEQUENT = 1e-3  # Lower LR to prevent drift and preserve historical memory
BATCH_SIZE = 256
CALIB_SAMPLE_SIZE = 1024

ALPHA_INIT = 1.2
G_INIT = 1.0

# Structural variables matching target deep architecture
hidden_size = 784
depth = 9
activation_name = "tanh"
bias = False

# Algorithm Configuration
GLOBAL_THRESHOLD = 0.97
ORTHOG_METHOD = "qr"

CUSTOM_MASKING_FN = None
EVAL_EVERY_N_BATCHES = 100

# --- 2. OPTIMIZED DATA LOADING ---
print(f"Initializing VRAM memory pinning pipeline on device: {DEVICE}")
mnist_train = datasets.MNIST(
    "../data", train=True, download=True, transform=transforms.ToTensor()
)
mnist_test = datasets.MNIST(
    "../data", train=False, download=True, transform=transforms.ToTensor()
)

train_imgs, train_lbls = get_gpu_data(mnist_train)
test_imgs_raw, test_lbls = get_gpu_data(mnist_test)


# --- 3. CORE BACKGROUND SWEEP PIPELINE ---
for current_seed in RUN_SEEDS:
    print("\n" + "=" * 80)
    print(f"LAUNCHING PARAMETER SWEEP FOR RANDOM EXPERIMENTAL SEED [s={current_seed}]")
    print("=" * 80)

    set_seed(current_seed)
    task_permutations = generate_permutations(num_tasks=NUM_TASKS, seed=current_seed)

    model = GeneralMLP(784, hidden_size, 10, depth, activation_name, bias=bias).to(
        DEVICE
    )
    model = apply_heavy_tailed_init(
        model=model, alpha=ALPHA_INIT, g=G_INIT, seed=current_seed
    )

    optimizer = optim.SGD(model.parameters(), lr=LR_TASK_0)
    criterion = nn.CrossEntropyLoss()
    tracker = CLMetricsTracker(max_tasks=NUM_TASKS)

    gpm = GPM(variance_threshold=GLOBAL_THRESHOLD, orthog_method=ORTHOG_METHOD)

    linear_layer_info = [
        (name + ".weight", module)
        for name, module in model.named_modules()
        if isinstance(module, nn.Linear)
    ]

    total_steps = 0

    print(f"Architecture Depth: {depth} layers | Hidden Width: {hidden_size} neurons")
    print(
        f"GPM Mode: {'Standard GPM' if CUSTOM_MASKING_FN is None else 'Custom Masked GPM'}"
    )
    print(
        f"Orthogonalization Method: {ORTHOG_METHOD.upper()} | Target Energy: {GLOBAL_THRESHOLD * 100:.1f}%"
    )
    print("-" * 80)

    # --- 4. MAIN CONTINUAL LEARNING STREAM ---
    for t_idx in range(NUM_TASKS):
        # Dynamically set Learning Rate: Task 0 vs Subsequent Tasks
        current_lr = LR_TASK_0 if t_idx == 0 else LR_SUBSEQUENT
        for param_group in optimizer.param_groups:
            param_group["lr"] = current_lr

        current_perm = task_permutations[t_idx]

        tx = train_imgs[:, current_perm]
        ty = train_lbls

        print(
            f"\n--- Task {t_idx:02d}/{NUM_TASKS:02d} | Seed: {current_seed} | Active LR: {current_lr:.1e} ---"
        )

        for epoch in range(EPOCHS_PER_TASK):
            # ... rest of your epoch loop stays identical ...
            model.train()
            indices = torch.randperm(len(tx))

            for i in range(0, len(tx), BATCH_SIZE):
                batch_idx = indices[i : i + BATCH_SIZE]
                bx, by = tx[batch_idx], ty[batch_idx]

                optimizer.zero_grad()

                output = model(bx)
                loss_current = criterion(output, by)
                loss_current.backward()

                gpm.project_model_gradients(model)
                optimizer.step()

                # Snapshot logging evaluation cycle
                if total_steps % EVAL_EVERY_N_BATCHES == 0:
                    current_accs = []
                    model.eval()
                    with torch.no_grad():
                        for eval_t_idx in range(t_idx + 1):
                            eval_perm = task_permutations[eval_t_idx]
                            test_x = test_imgs_raw[:, eval_perm]

                            outputs = model(test_x[:1000])
                            preds = outputs.argmax(dim=1)
                            acc = (preds == test_lbls[:1000]).float().mean().item()
                            current_accs.append(acc)

                    # Log task accuracy along with per-layer basis ranks and total rank
                    tracker.log(
                        step=total_steps,
                        acc_list=current_accs,
                        basis_rank=gpm.get_basis_ranks(),
                        total_basis_rank=gpm.get_total_basis_rank(),
                    )

                    acc_report = " | ".join(
                        [
                            f"T{j}: {current_accs[j] * 100:.1f}%"
                            for j in range(t_idx + 1)
                        ]
                    )
                    print(
                        f"Seed {current_seed} | Step {total_steps:04d} (Epoch {epoch}) -> {acc_report} | Total Basis Rank: {gpm.get_total_basis_rank()}"
                    )

                total_steps += 1

        # --- 5. POST-TASK CALIBRATION STEP ---
        model.eval()
        calib_indices = torch.randperm(len(tx))[:CALIB_SAMPLE_SIZE]
        calib_images = tx[calib_indices]

        with torch.no_grad():
            layer_inputs_dict = model.get_layer_inputs(calib_images)

        for (weight_param_name, _), (layer_key, layer_input) in zip(
            linear_layer_info, layer_inputs_dict.items()
        ):
            added_rank = gpm.update_basis(
                layer_id=weight_param_name,
                live_activations=layer_input,
                masking_fn=CUSTOM_MASKING_FN,
            )

        print(
            f"Task {t_idx} complete. Subspace memory updated. Current total basis rank: {gpm.get_total_basis_rank()}"
        )

    # --- 6. END-OF-TRAINING SNAPSHOT & SERIALIZATION ---
    output_dir = Path("./checkpoints")
    final_calib_indices = torch.randperm(len(tx))[:CALIB_SAMPLE_SIZE]
    final_batch = tx[final_calib_indices]

    # snapshot_file = save_physics_snapshot(
    #     model=model,
    #     input_batch=final_batch,
    #     output_dir=output_dir,
    #     t_idx=NUM_TASKS - 1,
    #     epoch=EPOCHS_PER_TASK - 1,
    #     alpha=ALPHA_INIT,
    #     g=G_INIT,
    # )
    # print(f"End-of-training physics snapshot successfully saved to: {snapshot_file}")

    output_filename = f"schedule_gpm_a{ALPHA_INIT}_run_s{current_seed}.csv"
    tracker.save_to_csv(output_filename)
    print(
        f"Sweep for seed {current_seed} serialized successfully to {output_filename}.\n"
    )

print("\n" + "=" * 80)
print("ALL MULTI-SEED PARAMETER CONVERSIONS COMPLETED IN BACKGROUND POOL.")
print("=" * 80)

In [ ]:
import os
from pathlib import Path

import torch
from torch import nn, optim
from torchvision import datasets, transforms

# --- 1. GLOBAL EXPERIMENT HYPERPARAMETERS ---
FIXED_SEED = 0
NUM_TASKS = 20
EPOCHS_PER_TASK = 5

# Learning Rate across tasks
LEARNING_RATE = 1e-3

BATCH_SIZE = 256
CALIB_SAMPLE_SIZE = 1024

# Phase Diagram Grid Parameters
ALPHA_VALS = np.round(np.arange(1.0, 2.01, 0.1), 2)  # 1.0 to 2.0 (11 steps)
G_VALS = np.round(np.arange(0.5, 3.01, 0.25), 2)  # 0.5 to 3.0 (11 steps)

# Testing
# ALPHA_VALS = [1.2, 2.0]  # Reduced for testing
# G_VALS = [1.0]

# Structural variables matching target deep architecture
hidden_size = 784
depth = 9
activation_name = "tanh"
bias = False

# Algorithm Configuration
GLOBAL_THRESHOLD = 0.97
ORTHOG_METHOD = "qr"

CUSTOM_MASKING_FN = None
EVAL_EVERY_N_BATCHES = 100

# --- 2. OUTPUT DIRECTORIES SETUP ---
BASE_DIR = Path("./phase_sweep")
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
RESULTS_DIR = BASE_DIR / "results"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory initialized: {BASE_DIR.resolve()}")
print(f"  ├── Checkpoints -> {CHECKPOINT_DIR.resolve()}")
print(f"  └── CSV Results -> {RESULTS_DIR.resolve()}")

# --- 3. OPTIMIZED DATA LOADING ---
print(f"\nInitializing VRAM memory pinning pipeline on device: {DEVICE}")
mnist_train = datasets.MNIST(
    "../data", train=True, download=True, transform=transforms.ToTensor()
)
mnist_test = datasets.MNIST(
    "../data", train=False, download=True, transform=transforms.ToTensor()
)

train_imgs, train_lbls = get_gpu_data(mnist_train)
test_imgs_raw, test_lbls = get_gpu_data(mnist_test)


# --- 4. CORE 2D GRID SWEEP PIPELINE (alpha x g) ---
total_runs = len(ALPHA_VALS) * len(G_VALS)
run_counter = 0

for alpha_init in ALPHA_VALS:
    for g_init in G_VALS:
        alpha_init = float(alpha_init)
        g_init = float(g_init)
        run_counter += 1

        print("\n" + "=" * 80)
        print(
            f"LAUNCHING GRID RUN [{run_counter:03d}/{total_runs:03d}] -> "
            f"ALPHA = {alpha_init:.2f} | GAIN (g) = {g_init:.2f}"
        )
        print("=" * 80)

        set_seed(FIXED_SEED)
        task_permutations = generate_permutations(
            num_tasks=NUM_TASKS, seed=FIXED_SEED
        )

        # Initialize Model & Apply Heavy-Tailed Initialization
        model = GeneralMLP(
            784, hidden_size, 10, depth, activation_name, bias=bias
        ).to(DEVICE)
        model = apply_heavy_tailed_init(
            model=model, alpha=alpha_init, g=g_init, seed=FIXED_SEED
        )

        optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)
        criterion = nn.CrossEntropyLoss()
        tracker = CLMetricsTracker(max_tasks=NUM_TASKS)

        gpm = GPM(
            variance_threshold=GLOBAL_THRESHOLD, orthog_method=ORTHOG_METHOD
        )

        linear_layer_info = [
            (name + ".weight", module)
            for name, module in model.named_modules()
            if isinstance(module, nn.Linear)
        ]

        total_steps = 0

        # --- 5. MAIN CONTINUAL LEARNING STREAM ---
        for t_idx in range(NUM_TASKS):
            current_perm = task_permutations[t_idx]

            tx = train_imgs[:, current_perm]
            ty = train_lbls

            for epoch in range(EPOCHS_PER_TASK):
                model.train()
                indices = torch.randperm(len(tx))

                for i in range(0, len(tx), BATCH_SIZE):
                    batch_idx = indices[i : i + BATCH_SIZE]
                    bx, by = tx[batch_idx], ty[batch_idx]

                    optimizer.zero_grad()

                    output = model(bx)
                    loss_current = criterion(output, by)
                    loss_current.backward()

                    gpm.project_model_gradients(model)
                    optimizer.step()

                    # Evaluation cycle
                    if total_steps % EVAL_EVERY_N_BATCHES == 0:
                        current_accs = []
                        model.eval()
                        with torch.no_grad():
                            for eval_t_idx in range(t_idx + 1):
                                eval_perm = task_permutations[eval_t_idx]
                                test_x = test_imgs_raw[:, eval_perm]

                                outputs = model(test_x[:1000])
                                preds = outputs.argmax(dim=1)
                                acc = (
                                    (preds == test_lbls[:1000])
                                    .float()
                                    .mean()
                                    .item()
                                )
                                current_accs.append(acc)

                        tracker.log(
                            step=total_steps,
                            acc_list=current_accs,
                            basis_rank=gpm.get_basis_ranks(),
                            total_basis_rank=gpm.get_total_basis_rank(),
                        )

                    total_steps += 1

            # --- POST-TASK CALIBRATION STEP ---
            model.eval()
            calib_indices = torch.randperm(len(tx))[:CALIB_SAMPLE_SIZE]
            calib_images = tx[calib_indices]

            with torch.no_grad():
                layer_inputs_dict = model.get_layer_inputs(calib_images)

            for (weight_param_name, _), (layer_key, layer_input) in zip(
                linear_layer_info, layer_inputs_dict.items()
            ):
                added_rank = gpm.update_basis(
                    layer_id=weight_param_name,
                    live_activations=layer_input,
                    masking_fn=CUSTOM_MASKING_FN,
                )

            # --- 5.5 FINAL EVALUATION LOG AFTER ALL TASKS & CALIBRATIONS ---
            model.eval()
            final_accs = []
            with torch.no_grad():
                for eval_t_idx in range(NUM_TASKS):
                    eval_perm = task_permutations[eval_t_idx]
                    test_x = test_imgs_raw[:, eval_perm]

                    outputs = model(test_x[:1000])
                    preds = outputs.argmax(dim=1)
                    acc = (preds == test_lbls[:1000]).float().mean().item()
                    final_accs.append(acc)

            # Log the absolute final post-calibration state into the CSV tracker
            tracker.log(
                step=total_steps,
                acc_list=final_accs,
                basis_rank=gpm.get_basis_ranks(),
                total_basis_rank=gpm.get_total_basis_rank(),
            )

            print(
                f"Task {t_idx:02d} complete | Current Total Basis Rank: {gpm.get_total_basis_rank()}"
            )

        # --- 6. SERIALIZATION (CHECKPOINT & RESULT CSV) ---
        run_id_str = f"a{alpha_init:.2f}_g{g_init:.2f}"

        # 1. Save final model checkpoint
        checkpoint_filepath = CHECKPOINT_DIR / f"snapshot_{run_id_str}.pt"
        torch.save(
            {
                "state_dict": model.state_dict(),
                "metadata": {
                    "alpha": alpha_init,
                    "g": g_init,
                    "seed": FIXED_SEED,
                    "total_basis_rank": gpm.get_total_basis_rank(),
                    "num_tasks": NUM_TASKS,
                    "depth": depth,
                    "hidden_size": hidden_size,
                },
            },
            checkpoint_filepath,
        )

        # 2. Save tracking metrics CSV
        results_filepath = RESULTS_DIR / f"results_{run_id_str}.csv"
        tracker.save_to_csv(results_filepath)

        print(
            f"Saved Checkpoint -> {checkpoint_filepath}\n"
            f"Saved Results CSV -> {results_filepath}"
        )

print("\n" + "=" * 80)
print("ALL PHASE DIAGRAM GRID SWEEPS COMPLETED SUCCESSFULLY.")
print("=" * 80)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- 1. CONFIGURATION & PATHS ---
RESULTS_DIR = Path("./phase_sweep/results")

# Match the exact grid parameters used in your sweep
ALPHA_VALS = np.round(np.arange(1.0, 2.01, 0.1), 2)  # 1.0 to 2.0 (11 steps)
G_VALS = np.round(np.arange(0.5, 3.01, 0.25), 2)  # 0.5 to 3.0 (11 steps)

NUM_ALPHA = len(ALPHA_VALS)
NUM_G = len(G_VALS)

# Mapping indices for array insertion
alpha_to_idx = {a: i for i, a in enumerate(ALPHA_VALS)}
g_to_idx = {g: j for j, g in enumerate(G_VALS)}

# Matrices configured for FLIPPED AXES: Rows = g (vertical), Columns = alpha (horizontal)
matrix_final_acc = np.full((NUM_G, NUM_ALPHA), np.nan)
matrix_task20_auc = np.full((NUM_G, NUM_ALPHA), np.nan)
matrix_hidden_rank = np.full((NUM_G, NUM_ALPHA), np.nan)


# --- 2. DATA AGGREGATION MATCHING YOUR CSV HEADERS ---
if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"Results directory not found at {RESULTS_DIR}")

csv_files = list(RESULTS_DIR.glob("results_a*_g*.csv"))
print(
    f"Found {len(csv_files)} result CSV files. Parsing matching headers..."
)

for csv_path in csv_files:
    # Parse alpha and g from filename (e.g., results_a1.20_g1.50.csv)
    filename = csv_path.stem
    parts = filename.split("_")
    alpha_val = float(parts[1].replace("a", ""))
    g_val = float(parts[2].replace("g", ""))

    if alpha_val not in alpha_to_idx or g_val not in g_to_idx:
        continue

    # Note the flipped index order: row = g, col = alpha
    row_g = g_to_idx[g_val]
    col_a = alpha_to_idx[alpha_val]

    df = pd.read_csv(csv_path)

    # Match exact accuracy columns: task_0_acc ... task_19_acc
    acc_cols = [f"task_{t}_acc" for t in range(20) if f"task_{t}_acc" in df.columns]

    # A. Final Average Accuracy (Mean across all 20 tasks in the absolute final state)
    if acc_cols:
        final_row = df.iloc[-1]
        matrix_final_acc[row_g, col_a] = final_row[acc_cols].mean()

    # B. Final Task (Task 19) Plasticity / Trajectory AUC
    # Measures the normalized Area Under the Curve (AUC) for Task 19 during its training phase
    if "task_19_acc" in df.columns:
        t20_values = df["task_19_acc"].dropna()
        if len(t20_values) > 0:
            # Take the final evaluated accuracy score for Task 20
            matrix_task20_auc[row_g, col_a] = t20_values.iloc[-1]

    # C. Hidden Layer Reserved Rank (Excludes Layer 1: basis_rank_features.0.weight)
    if (
        "total_basis_rank" in df.columns
        and "basis_rank_features.0.weight" in df.columns
    ):
        final_total_rank = df["total_basis_rank"].iloc[-1]
        final_layer1_rank = df["basis_rank_features.0.weight"].iloc[-1]
        matrix_hidden_rank[row_g, col_a] = final_total_rank - final_layer1_rank
    elif "total_basis_rank" in df.columns:
        matrix_hidden_rank[row_g, col_a] = df["total_basis_rank"].iloc[-1]

print("Grid aggregation complete!")


# --- 3. FLIPPED AXES PHASE DIAGRAM PLOTTING ROUTINE ---
def plot_phase_heatmap(
    matrix_data, title, cbar_label, cmap="viridis", fmt=".2f"
):
    plt.figure(figsize=(10, 7.5))

    # Heatmap setup: Y-axis = g (vertical), X-axis = alpha (horizontal)
    ax = sns.heatmap(
        matrix_data,
        xticklabels=ALPHA_VALS,
        yticklabels=G_VALS,
        annot=True,
        fmt=fmt,
        cmap=cmap,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": cbar_label},
        annot_kws={"size": 8.5, "weight": "bold"},
    )

    # Invert Y-axis so g increases upwards (standard physics convention)
    ax.invert_yaxis()

    plt.xlabel(
        r"Tail Exponent ($\alpha$)", fontweight="bold", fontsize=12
    )
    plt.ylabel(
        r"Initialization Gain ($g$)", fontweight="bold", fontsize=12
    )
    plt.title(title, fontweight="bold", fontsize=13, pad=14)

    plt.tight_layout()
    plt.show()


# --- 4. GENERATE THE THREE PHASE DIAGRAMS ---

# 1. Final Average Accuracy Phase Diagram
plot_phase_heatmap(
    matrix_final_acc,
    title="Phase Diagram: Final 20-Task Average Test Accuracy\n"
    "Maps Continual Learning Performance Across Parameter Space",
    cbar_label="Mean Test Accuracy",
    cmap="magma",
    fmt=".3f",
)

# 2. Final Task (Task 20) Plasticity / Trajectory AUC Phase Diagram
plot_phase_heatmap(
    matrix_task20_auc,
    title="Phase Diagram: Task 20 Accuracy\n"
    "Identifies Capacity Exhaustion vs. Retained Learning Ability",
    cbar_label="Task 20 Final Accuracy",
    cmap="plasma",
    fmt=".3f",
)

# 3. Hidden Layer Reserved Basis Rank Phase Diagram (Excludes Layer 1)
plot_phase_heatmap(
    matrix_hidden_rank,
    title="Phase Diagram: Hidden Layer Reserved Basis Rank ($K_{\\text{hidden}}$)\n"
    "Quantifies Subspace Compression (Excludes Invariant Layer 1)",
    cbar_label="Hidden Layer Basis Vectors",
    cmap="viridis_r",  # Reversed so lower rank (higher compression) stands out
    fmt=".0f",
)

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- 1. EXTRACT DEPTH COMPRESSION RATIO (K_last_hidden / K_first_hidden) ---
matrix_compression_ratio = np.full((NUM_ALPHA, NUM_G), np.nan)

for csv_path in csv_files:
    filename = csv_path.stem
    parts = filename.split("_")
    alpha_val = float(parts[1].replace("a", ""))
    g_val = float(parts[2].replace("g", ""))

    if alpha_val not in alpha_to_idx or g_val not in g_to_idx:
        continue

    row_a = alpha_to_idx[alpha_val]
    col_g = g_to_idx[g_val]

    df = pd.read_csv(csv_path)

    # Identify layer rank columns
    layer_cols = [
        c
        for c in df.columns
        if c.startswith("basis_rank_") and c.endswith(".weight")
    ]

    if len(layer_cols) >= 2:
        # layer_cols[1] = First non-input hidden layer
        # layer_cols[-1] = Last hidden layer (or pre-classifier layer)
        first_hidden_rank = df[layer_cols[1]].iloc[-1]
        last_hidden_rank = df[layer_cols[-1]].iloc[-1]

        # Compute Depth Compression Ratio: C_depth = K_last / K_first
        # Ratio < 1.0 = Contractive / Compression Phase
        # Ratio > 1.0 = Expansive Phase
        if first_hidden_rank > 0:
            matrix_compression_ratio[row_a, col_g] = (
                last_hidden_rank / first_hidden_rank
            )


# --- 2. DIVERGING HEATMAP ROUTINE WITH FIXED WHITE POINT AT 1.0 ---
def plot_ratio_phase_heatmap(
    matrix_data, title, cbar_label, cmap="seismic", fmt=".2f"
):
    plt.figure(figsize=(9.5, 7.5))

    # Mask valid numeric values to compute dynamic bounds around center=1.0
    valid_data = matrix_data[~np.isnan(matrix_data)]
    vmin = np.min(valid_data)
    vmax = np.max(valid_data)

    # TwoSlopeNorm locks the neutral center of the colormap strictly at vcenter=1.0
    norm = mcolors.TwoSlopeNorm(vcenter=1.0, vmin=vmin, vmax=vmax)

    ax = sns.heatmap(
        matrix_data,
        xticklabels=ALPHA_VALS,
        yticklabels=G_VALS,
        annot=True,
        fmt=fmt,
        cmap=cmap,
        norm=norm,  # Enforces 1.0 = White
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": cbar_label},
        annot_kws={"size": 8.5, "weight": "bold"},
    )

    # Invert Y-axis so alpha increases upwards (standard physics convention)
    ax.invert_yaxis()

    plt.xlabel(r"Tail Exponent ($\alpha$)", fontweight="bold", fontsize=12)
    plt.ylabel(
        r"Initialization Gain ($g$)", fontweight="bold", fontsize=12
    )
    plt.title(title, fontweight="bold", fontsize=13, pad=14)

    plt.tight_layout()
    plt.show()


# --- 3. PLOT COMPRESSION RATIO PHASE DIAGRAM ---
plot_ratio_phase_heatmap(
    matrix_compression_ratio,
    title="Phase Diagram: Depth Compression Ratio "
    r"($C_{\text{depth}} = K_{\text{last\_hidden}} / K_{\text{first\_hidden}}$)"
    "\nBlue = Compression Phase ($C < 1.0$) | White = Neutral ($C = 1.0$) | Red = Expansion Phase ($C > 1.0$)",
    cbar_label=r"Depth Compression Ratio ($C_{\text{depth}}$)",
    cmap="seismic",
    fmt=".2f",
)

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- 1. CONFIGURATION ---
GAUSSIAN_CSV = "gpm_a2.0_run_s0.csv"  # Standard Gaussian initialization run
HEAVY_TAIL_CSV = "gpm_a1.2_run_s0.csv"  # Heavy-Tailed initialization run
NUM_TASKS = 20


# --- 2. EXTRACTOR FUNCTION FOR A SINGLE CSV FILE ---
def extract_run_metrics(filepath, num_tasks=20):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Could not find target CSV file at: '{filepath}'")

    df = pd.read_csv(filepath)
    num_entries = len(df)

    # Sort chronological accuracy columns
    task_acc_cols = [
        c for c in df.columns if c.startswith("task_") and c.endswith("_acc")
    ]
    task_acc_cols = sorted(task_acc_cols, key=lambda x: int(x.split("_")[1]))

    # Determine active task footprint per row to find task completion boundaries
    active_tasks_per_step = np.array(
        [df.iloc[idx][task_acc_cols].notna().sum() for idx in range(num_entries)]
    )

    # Detect transition step indices where each task finishes training
    transition_indices = []
    current_num = active_tasks_per_step[0]
    for idx in range(1, num_entries):
        if active_tasks_per_step[idx] > current_num:
            transition_indices.append(idx - 1)
            current_num = active_tasks_per_step[idx]
    transition_indices.append(num_entries - 1)

    if len(transition_indices) != num_tasks:
        print(
            f"Warning: Boundary mismatch in {filepath}. Detected {len(transition_indices)} boundaries instead of {num_tasks}"
        )

    # Determine the step intervals for each task
    task_step_intervals = []
    start_idx = 0
    for end_idx in transition_indices:
        task_step_intervals.append((start_idx, end_idx))
        start_idx = end_idx + 1

    # Storage arrays
    avg_acc_at_wrapup = []
    current_task_auc = []
    total_basis_rank = []
    layer_1_basis_rank = []

    # Detect available rank columns dynamically
    layer_1_col = "basis_rank_0" if "basis_rank_0" in df.columns else None
    if layer_1_col is None:
        layer_1_cols = [
            c
            for c in df.columns
            if "basis" in c and ("0" in c or "layer_0" in c or "fc1" in c)
        ]
        layer_1_col = layer_1_cols[0] if len(layer_1_cols) > 0 else None

    for t_idx, boundary_idx in enumerate(transition_indices):
        # 1. Global Average Accuracy across all active tasks
        row_accs = df.iloc[boundary_idx][task_acc_cols].values
        active_accs = row_accs[~pd.isna(row_accs)]
        avg_acc_at_wrapup.append(np.mean(active_accs) if len(active_accs) > 0 else 0.0)

        # 2. Plasticity AUC during training of Task t_idx
        start_step_idx, end_step_idx = task_step_intervals[t_idx]
        task_col = f"task_{t_idx}_acc"

        task_trajectory = (
            df.iloc[start_step_idx : end_step_idx + 1][task_col].dropna().values
        )
        steps = (
            df.iloc[start_step_idx : end_step_idx + 1]["step"]
            .iloc[: len(task_trajectory)]
            .values
        )

        if len(task_trajectory) > 1:
            auc_val = np.trapezoid(y=task_trajectory, x=steps) / (steps[-1] - steps[0])
        elif len(task_trajectory) == 1:
            auc_val = task_trajectory[0]
        else:
            auc_val = 0.0
        current_task_auc.append(auc_val)

        # 3. Total Basis Rank at task wrap-up
        if "total_basis_rank" in df.columns:
            total_rank_val = df.iloc[boundary_idx]["total_basis_rank"]
        else:
            rank_cols = [c for c in df.columns if c.startswith("basis_rank_")]
            total_rank_val = (
                df.iloc[boundary_idx][rank_cols].sum() if len(rank_cols) > 0 else np.nan
            )
        total_basis_rank.append(total_rank_val)

        # 4. Layer 1 Basis Rank at task wrap-up
        if layer_1_col and layer_1_col in df.columns:
            layer_1_val = df.iloc[boundary_idx][layer_1_col]
        else:
            layer_1_val = np.nan
        layer_1_basis_rank.append(layer_1_val)

    return {
        "avg_acc": avg_acc_at_wrapup,
        "auc": current_task_auc,
        "total_rank": total_basis_rank,
        "layer1_rank": layer_1_basis_rank,
    }


# --- 3. PROCESS BOTH RUNS ---
print(f"Extracting Gaussian baseline metrics from: {GAUSSIAN_CSV}")
gauss_data = extract_run_metrics(GAUSSIAN_CSV, NUM_TASKS)

print(f"Extracting Heavy-Tailed metrics from: {HEAVY_TAIL_CSV}")
ht_data = extract_run_metrics(HEAVY_TAIL_CSV, NUM_TASKS)


# --- 4. 2x2 COMPARATIVE VISUALIZATION ---
fig, axs = plt.subplots(2, 2, figsize=(15, 11), dpi=100)
tasks_axis = np.arange(1, NUM_TASKS + 1)

# Color and style definitions
gauss_color, ht_color = "crimson", "dodgerblue"
gauss_marker, ht_marker = "o", "s"

# [TOP LEFT] Global Cumulative Average Accuracy
axs[0, 0].plot(
    tasks_axis,
    gauss_data["avg_acc"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Gaussian",
)
axs[0, 0].plot(
    tasks_axis,
    ht_data["avg_acc"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[0, 0].set_title("1. Global Average Test Accuracy", fontsize=11, weight="bold")
axs[0, 0].set_xlabel("Task Index")
axs[0, 0].set_ylabel("Mean Accuracy across Learned Tasks")
axs[0, 0].set_xticks(tasks_axis)
axs[0, 0].grid(True, linestyle=":", alpha=0.5)
axs[0, 0].legend()

# [TOP RIGHT] Task Plasticity AUC
axs[0, 1].plot(
    tasks_axis,
    gauss_data["auc"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Gaussian",
)
axs[0, 1].plot(
    tasks_axis,
    ht_data["auc"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[0, 1].set_title(
    "2. Current Task Learning Curve AUC (Plasticity)", fontsize=11, weight="bold"
)
axs[0, 1].set_xlabel("Task Index")
axs[0, 1].set_ylabel("Normalized Training AUC")
axs[0, 1].set_xticks(tasks_axis)
axs[0, 1].grid(True, linestyle=":", alpha=0.5)
axs[0, 1].legend()

# [BOTTOM LEFT] Total Basis Rank
axs[1, 0].plot(
    tasks_axis,
    gauss_data["total_rank"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Gaussian",
)
axs[1, 0].plot(
    tasks_axis,
    ht_data["total_rank"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[1, 0].set_title("3. Cumulative Total Basis Rank", fontsize=11, weight="bold")
axs[1, 0].set_xlabel("Task Index")
axs[1, 0].set_ylabel("Total Reserved Basis Vectors")
axs[1, 0].set_xticks(tasks_axis)
axs[1, 0].grid(True, linestyle=":", alpha=0.5)
axs[1, 0].legend()

# [BOTTOM RIGHT] Layer 1 Basis Rank
axs[1, 1].plot(
    tasks_axis,
    gauss_data["layer1_rank"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Gaussian",
)
axs[1, 1].plot(
    tasks_axis,
    ht_data["layer1_rank"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[1, 1].set_title("4. Layer 1 Basis Rank", fontsize=11, weight="bold")
axs[1, 1].set_xlabel("Task Index")
axs[1, 1].set_ylabel("Layer 1 Reserved Basis Vectors")
axs[1, 1].set_xticks(tasks_axis)
axs[1, 1].grid(True, linestyle=":", alpha=0.5)
axs[1, 1].legend()

plt.tight_layout()
plt.savefig("gpm_gaussian_vs_heavytail_comparison.pdf", bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path

import torch
from torch import nn
from torchvision import datasets, transforms

# --- 1. CONFIGURATION & PATHS ---
# Update paths to point to your respective snapshot files
PATH_GAUSSIAN = Path("./checkpoints/snapshot_A2.0_T20_E4.pt")
PATH_HEAVY_TAILED = Path("./checkpoints/snapshot_A1.2_T20_E4.pt")

NUM_TASKS = 20
BATCH_SIZE = 1024  # Size of the evaluation batch per task

# Architecture hyper-parameters
HIDDEN_SIZE = 784
DEPTH = 9
ACTIVATION_NAME = "tanh"
BIAS = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- 2. EXTRACTION HELPER FUNCTION ---
def extract_model_and_preactivations(
    snapshot_path: Path,
    test_imgs_raw: torch.Tensor,
    num_tasks: int = 20,
    batch_size: int = 1024,
    device: torch.device = DEVICE,
):
    """Loads a snapshot checkpoint, reconstructs the model, and extracts

    pre-activations across all tasks.
    """
    if not snapshot_path.exists():
        raise FileNotFoundError(f"Snapshot not found at: {snapshot_path}")

    print(f"\n--- Processing Snapshot: {snapshot_path.name} ---")
    snapshot = torch.load(snapshot_path, map_location=device)
    metadata = snapshot.get("metadata", {})
    seed = metadata.get("seed", 0)

    print(
        f"Metadata -> Task: {metadata.get('task')}, Epoch: {metadata.get('epoch')}, Seed: {seed}"
    )

    # Reconstruct architecture and load weights
    model = GeneralMLP(
        input_size=784,
        hidden_size=HIDDEN_SIZE,
        num_classes=10,
        depth=DEPTH,
        activation_name=ACTIVATION_NAME,
        bias=BIAS,
    ).to(device)

    model.load_state_dict(snapshot["state_dict"])
    model.eval()

    # Regenerate task permutations matching the snapshot's seed
    set_seed(seed)
    task_permutations = generate_permutations(num_tasks=num_tasks, seed=seed)

    # Extract pre-activations per layer per task
    task_pre_acts = []
    with torch.no_grad():
        for t_idx in range(num_tasks):
            perm = task_permutations[t_idx]
            task_batch = test_imgs_raw[:batch_size, perm]

            pre_acts = model.get_pre_activations(task_batch)

            if isinstance(pre_acts, dict):
                layer_keys = [k for k in pre_acts.keys() if k != "classifier"]
                layer_list = [pre_acts[k] for k in layer_keys]
                if "classifier" in pre_acts:
                    layer_list.append(pre_acts["classifier"])
            else:
                layer_list = pre_acts

            task_pre_acts.append(layer_list)

    print(f"Extraction complete for {snapshot_path.name}.")

    return {
        "model": model,
        "task_pre_acts": task_pre_acts,
        "metadata": metadata,
        "seed": seed,
    }


# --- 3. MAIN EXECUTION PIPELINE ---

# A. Prepare Data (Loaded once for both runs)
mnist_test = datasets.MNIST(
    "../data", train=False, download=True, transform=transforms.ToTensor()
)
test_imgs_raw, _ = get_gpu_data(mnist_test)  # Unpermuted GPU images [N, 784]

# B. Process Both Snapshots
run_gaussian = extract_model_and_preactivations(
    snapshot_path=PATH_GAUSSIAN,
    test_imgs_raw=test_imgs_raw,
    num_tasks=NUM_TASKS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

run_ht = extract_model_and_preactivations(
    snapshot_path=PATH_HEAVY_TAILED,
    test_imgs_raw=test_imgs_raw,
    num_tasks=NUM_TASKS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

# C. Consolidated Output Dictionary ready for comparative plotting
runs_data = {"gaussian": run_gaussian, "heavy_tailed": run_ht}

print(
    "\nExtraction successfully completed for both Gaussian and Heavy-Tailed runs!"
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib import cm


def convert_pre_to_post_activations(task_pre_acts, act_fn=torch.tanh):
    """Converts a nested list of pre-activations [num_tasks][num_layers]
    into 1D mean absolute post-activations [hidden_dim] per task and layer.
    """
    task_post_acts = []
    for t_idx in range(len(task_pre_acts)):
        layer_posts = []
        for layer_pre in task_pre_acts[t_idx]:
            post_act = act_fn(layer_pre)
            mean_abs_act = torch.abs(post_act).mean(dim=0)
            layer_posts.append(mean_abs_act.cpu().numpy())
        task_post_acts.append(layer_posts)
    return task_post_acts


def compute_all_pairs_energy_jaccard(task_post_acts, layer_idx, energy_fractions):
    """Computes Energy-Gated Soft Jaccard across all N*(N-1)/2 task pairs
    for a given layer index.
    """
    num_tasks = len(task_post_acts)
    task_acts = [task_post_acts[t][layer_idx] for t in range(num_tasks)]

    pair_indices = [(i, j) for i in range(num_tasks) for j in range(i + 1, num_tasks)]
    num_pairs = len(pair_indices)

    all_jaccard_curves = np.zeros((num_pairs, len(energy_fractions)))
    all_k_curves = np.zeros((num_pairs, len(energy_fractions)))

    print(f"Processing {num_pairs} pairwise comparisons for Layer {layer_idx + 1}...")

    for p_idx, (i, j) in enumerate(pair_indices):
        act_A, act_B = task_acts[i], task_acts[j]

        energy_A, energy_B = act_A**2, act_B**2
        total_E_A, total_E_B = np.sum(energy_A), np.sum(energy_B)

        sort_idx_A = np.argsort(energy_A)[::-1]
        sort_idx_B = np.argsort(energy_B)[::-1]

        cumsum_E_A = np.cumsum(energy_A[sort_idx_A]) / (total_E_A + 1e-12)
        cumsum_E_B = np.cumsum(energy_B[sort_idx_B]) / (total_E_B + 1e-12)

        for e_idx, target_E in enumerate(energy_fractions):
            k_A = min(np.searchsorted(cumsum_E_A, target_E) + 1, len(act_A))
            k_B = min(np.searchsorted(cumsum_E_B, target_E) + 1, len(act_B))

            gated_A = np.zeros_like(act_A)
            gated_B = np.zeros_like(act_B)

            gated_A[sort_idx_A[:k_A]] = act_A[sort_idx_A[:k_A]]
            gated_B[sort_idx_B[:k_B]] = act_B[sort_idx_B[:k_B]]

            num = np.sum(np.minimum(gated_A, gated_B))
            den = np.sum(np.maximum(gated_A, gated_B))

            all_jaccard_curves[p_idx, e_idx] = num / den if den > 0 else 0.0
            all_k_curves[p_idx, e_idx] = (k_A + k_B) / 2.0

    return all_jaccard_curves, all_k_curves


def run_bootstrap_ci(data_matrix, num_bootstraps=1000, ci_level=95):
    """Computes bootstrap mean and percentile confidence intervals along axis 0."""
    num_pairs, num_points = data_matrix.shape
    boot_means = np.zeros((num_bootstraps, num_points))

    rng = np.random.default_rng(seed=42)
    for b in range(num_bootstraps):
        boot_indices = rng.choice(num_pairs, size=num_pairs, replace=True)
        boot_means[b, :] = np.mean(data_matrix[boot_indices, :], axis=0)

    lower_p = (100 - ci_level) / 2.0
    upper_p = 100 - lower_p

    mean_curve = np.mean(data_matrix, axis=0)
    ci_lower = np.percentile(boot_means, lower_p, axis=0)
    ci_upper = np.percentile(boot_means, upper_p, axis=0)

    return mean_curve, ci_lower, ci_upper


# --- EXECUTION & MULTI-LAYER PLOTTING ROUTINE ---
ACTIVATION_FN = torch.tanh  # Or torch.relu for ReLU runs
task_post_acts = convert_pre_to_post_activations(task_pre_acts, act_fn=ACTIVATION_FN)

# 1. SELECT TARGET LAYERS TO COMPARE ACROSS DEPTH (0-indexed)
# For a 9-layer MLP, selecting input, early, mid, late, and classifier layers
TARGET_LAYERS = [4]
SHOW_CONFIDENCE_INTERVALS = True  # Toggle shaded CI bands on/off for readability

energy_fractions = np.linspace(0.02, 1.0, 99)
x_axis = energy_fractions * 100

# Generate color spectrum from blue (shallow) to red/purple (deep)
colors = cm.plasma(np.linspace(0.1, 0.9, len(TARGET_LAYERS)))

fig, ax1 = plt.subplots(figsize=(12, 7), dpi=100)
ax2 = ax1.twinx()

for idx, l_idx in enumerate(TARGET_LAYERS):
    # Process 190 task pairs for layer l_idx
    j_matrix, k_matrix = compute_all_pairs_energy_jaccard(
        task_post_acts, l_idx, energy_fractions
    )

    # Perform Bootstrapping
    j_mean, j_low, j_high = run_bootstrap_ci(j_matrix, num_bootstraps=1000)
    k_mean, k_low, k_high = run_bootstrap_ci(k_matrix, num_bootstraps=1000)

    color = colors[idx]
    layer_label = f"Layer {l_idx + 1}"

    # Primary Axis: Soft Jaccard Overlap
    ax1.plot(
        x_axis,
        j_mean,
        color=color,
        linewidth=2.5,
        label=f"{layer_label} Jaccard",
    )

    if SHOW_CONFIDENCE_INTERVALS:
        ax1.fill_between(x_axis, j_low, j_high, color=color, alpha=0.1)

    # Secondary Axis: Active Neurons (k)
    ax2.plot(
        x_axis,
        k_mean,
        color=color,
        linestyle="--",
        linewidth=1.5,
        alpha=0.7,
        label=f"{layer_label} Active k",
    )

# Axis Formatting
ax1.set_xlabel("Cumulative Task Activation Energy Mass (%)", fontsize=11, weight="bold")
ax1.set_ylabel("Continuous Soft Jaccard Overlap Score", fontsize=11, weight="bold")
ax1.set_xlim(0, 100)
ax1.set_ylim(0, 1.0)
ax1.grid(True, linestyle=":", alpha=0.6)

ax2.set_ylabel(
    "Mean Active Neurons Required (k)", color="gray", fontsize=11, weight="bold"
)

plt.title(
    "Depth-Wise Energy-Gated Soft Jaccard Evolution (190 Task Pairs)\nComparing Representation Compression & Overlap Across Layers",
    fontsize=12,
    weight="bold",
)

# Combine legends cleanly
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left", fontsize=9, ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import skdim
import torch
from scipy.special import digamma
from sklearn.neighbors import NearestNeighbors


# --- 1. MATHEMATICALLY ALIGNED ESTIMATORS ---
def compute_global_svd_rank(X, variance_threshold=0.95):
    """Computes global linear rank using SVD at a cumulative variance threshold."""
    # Center activations
    X_centered = X - np.mean(X, axis=0, keepdims=True)

    # Singular Value Decomposition
    _, S, _ = np.linalg.svd(X_centered, full_matrices=False)

    # Calculate cumulative variance fraction
    variance_explained = (S**2) / np.sum(S**2)
    cumsum_var = np.cumsum(variance_explained)

    # Find number of components to reach variance_threshold
    rank = np.searchsorted(cumsum_var, variance_threshold) + 1
    return min(rank, X.shape[1])


def compute_local_pca_id(X, k=30, variance_threshold=0.95):
    """Computes Local PCA ID across k-NN neighborhoods matching the 95% variance threshold."""
    # ver='ratio' forces lPCA to use cumulative variance thresholding (alphaRatio)
    lpca_model = skdim.id.lPCA(ver="ratio", alphaRatio=variance_threshold)

    # Fit pointwise across k-NN neighborhoods
    lpca_model.fit_pw(X, n_neighbors=k)

    # Average local tangent rank across all neighborhood balls
    return np.mean(lpca_model.dimension_pw_)


def compute_gride_id(X, k1=10, k2=20):
    """Computes non-linear Intrinsic Dimension using GRIDE (Generalized Ratio Estimator).

    MLE formula: d = [digamma(k2) - digamma(k1)] / mean(log(r_k2 / r_k1))
    """
    # Compute k2 nearest neighbors for each point
    nbrs = NearestNeighbors(n_neighbors=k2 + 1, algorithm="auto").fit(X)
    distances, _ = nbrs.kneighbors(X)

    # Extract distances to k1-th and k2-th neighbors
    r1 = distances[:, k1]
    r2 = distances[:, k2]

    # Filter out identical or zero distances
    valid_mask = (r1 > 1e-12) & (r2 > r1)
    if not np.any(valid_mask):
        return 0.0

    log_ratios = np.log(r2[valid_mask] / r1[valid_mask])
    mean_log_ratio = np.mean(log_ratios)

    # Digamma difference numerator
    digamma_diff = digamma(k2) - digamma(k1)

    return digamma_diff / mean_log_ratio


def compute_twonn_id(X):
    """Optional alternative: Native skdim TwoNN estimator (k1=1, k2=2)."""
    twonn = skdim.id.TwoNN()
    twonn.fit(X)
    return twonn.dimension_


# --- 2. EXECUTION ROUTINE ACROSS ALL LAYERS ---
TARGET_TASK_IDX = 1  # Task 0 for clean baseline geometry
VARIANCE_THRESHOLD = 0.95  # 95% energy mass threshold
ACTIVATION_FN = torch.tanh  # Post-activations

num_layers = len(task_pre_acts[TARGET_TASK_IDX])

global_svd_ranks = []
local_pca_ids = []
gride_ids = []

print(
    f"Evaluating Dimension Metrics for Task {TARGET_TASK_IDX} across {num_layers} layers..."
)

for l_idx in range(num_layers):
    layer_pre = task_pre_acts[TARGET_TASK_IDX][l_idx]

    # Convert to post-activations: [N_samples, hidden_dim]
    if isinstance(layer_pre, torch.Tensor):
        post_act_samples = ACTIVATION_FN(layer_pre).detach().cpu().numpy()
    else:
        post_act_samples = ACTIVATION_FN(torch.tensor(layer_pre)).numpy()

    print(
        f"--> Processing Layer {l_idx + 1}/{num_layers} (Matrix Shape: {post_act_samples.shape})..."
    )

    # 1. Global SVD Rank (95% Variance)
    g_svd = compute_global_svd_rank(
        post_act_samples, variance_threshold=VARIANCE_THRESHOLD
    )

    # 2. Local PCA ID (Local 95% Tangent Rank in 30-NN neighborhood)
    l_pca = compute_local_pca_id(
        post_act_samples, k=30, variance_threshold=VARIANCE_THRESHOLD
    )

    # 3. GRIDE ID (Noise-filtered non-linear manifold dimension)
    gride = compute_gride_id(post_act_samples, k1=10, k2=20)

    global_svd_ranks.append(g_svd)
    local_pca_ids.append(l_pca)
    gride_ids.append(gride)

    print(
        f"    Global SVD (95%): {g_svd} | Local PCA (95%): {l_pca:.1f} | GRIDE: {gride:.1f}"
    )


# --- 3. PLOTTING LAYERWISE COMPARISON CURVE ---
layers_x = np.arange(1, num_layers + 1)

plt.figure(figsize=(10, 6), dpi=100)

plt.plot(
    layers_x,
    global_svd_ranks,
    marker="o",
    color="darkred",
    linewidth=2.5,
    markersize=7,
    label=f"Global SVD Rank ({int(VARIANCE_THRESHOLD * 100)}% Variance)",
)

plt.plot(
    layers_x,
    local_pca_ids,
    marker="s",
    color="darkorange",
    linewidth=2.5,
    markersize=7,
    linestyle="--",
    label=f"Local PCA ID (Local {int(VARIANCE_THRESHOLD * 100)}% Tangent Rank)",
)

plt.plot(
    layers_x,
    gride_ids,
    marker="^",
    color="dodgerblue",
    linewidth=2.5,
    markersize=7,
    linestyle="-.",
    label="GRIDE ID (Non-Linear Manifold Dimension, k1=10, k2=20)",
)

plt.title(
    f"Layerwise Intrinsic Dimensionality & Linear Rank Profile\nTask {TARGET_TASK_IDX} Post-Activations | Depth-Wise Manifold Trajectory",
    fontsize=12,
    weight="bold",
)
plt.xlabel("Layer Index (Depth)", fontsize=11, weight="bold")
plt.ylabel("Estimated Dimension / Rank", fontsize=11, weight="bold")
plt.xticks(layers_x)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper right", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch


# --- 1. SUBSPACE EXTRACTION AND METRIC FUNCTIONS ---
def extract_task_basis(X, variance_threshold=0.95):
    """Extracts the top-k right singular vectors V_k representing the feature subspace

    that accounts for the given cumulative variance threshold.
    X shape: [N_samples, hidden_dim]
    """
    # Center activations across samples
    X_centered = X - np.mean(X, axis=0, keepdims=True)

    # Compute SVD: X = U @ S @ Vt
    _, S, Vt = np.linalg.svd(X_centered, full_matrices=False)

    # Compute required rank k
    var_explained = (S**2) / np.sum(S**2)
    cumsum_var = np.cumsum(var_explained)
    k = np.searchsorted(cumsum_var, variance_threshold) + 1
    k = min(k, X.shape[1])

    # Top-k feature basis vectors in ambient space: shape [hidden_dim, k]
    V_k = Vt[:k, :].T
    return V_k, X_centered


def compute_normalized_subspace_overlap(V_A, V_B):
    """Method A: Normalized Subspace Overlap Score ||V_A^T V_B||_F^2 / min(k_A, k_B)

    Bounded in [0.0, 1.0].
    """
    k_A, k_B = V_A.shape[1], V_B.shape[1]
    projection_matrix = V_A.T @ V_B  # [k_A, k_B]
    frobenius_sq = np.linalg.norm(projection_matrix, ord="fro") ** 2
    return frobenius_sq / min(k_A, k_B)


def compute_unexplained_variance_fraction(X_B_centered, V_A):
    """Method C: Fraction of Task B's energy that lies OUTSIDE Task A's subspace.

    Mirrors GPM's memory allocation check. Bounded in [0.0, 1.0].
    """
    # Project Task B onto Task A's basis subspace
    projected_B = X_B_centered @ V_A @ V_A.T
    explained_energy = np.linalg.norm(projected_B, ord="fro") ** 2
    total_energy = np.linalg.norm(X_B_centered, ord="fro") ** 2 + 1e-12

    unexplained_fraction = 1.0 - (explained_energy / total_energy)
    return max(0.0, unexplained_fraction)


# --- 2. EXECUTION ROUTINE ACROSS ALL LAYERS ---
VARIANCE_THRESHOLD = 0.95  # 95% energy threshold
ACTIVATION_FN = torch.tanh  # Post-activations

num_tasks = len(task_pre_acts)
num_layers = len(task_pre_acts[0])
task_pairs = list(itertools.combinations(range(num_tasks), 2))
num_pairs = len(task_pairs)

mean_overlaps_per_layer = []
mean_unexplained_per_layer = []

print(
    f"Evaluating Subspace Overlap across {num_layers} layers for {num_pairs} task pairs..."
)

for l_idx in range(num_layers):
    # 1. Extract post-activations and bases for all tasks at layer l_idx
    task_bases = []
    task_acts_centered = []

    for t_idx in range(num_tasks):
        layer_pre = task_pre_acts[t_idx][l_idx]
        if isinstance(layer_pre, torch.Tensor):
            post_act = ACTIVATION_FN(layer_pre).detach().cpu().numpy()
        else:
            post_act = ACTIVATION_FN(torch.tensor(layer_pre)).numpy()

        V_k, X_centered = extract_task_basis(
            post_act, variance_threshold=VARIANCE_THRESHOLD
        )
        task_bases.append(V_k)
        task_acts_centered.append(X_centered)

    # 2. Compute metrics across all 190 unique pairs (A, B)
    pair_overlaps = []
    pair_unexplained = []

    for t_A, t_B in task_pairs:
        V_A, V_B = task_bases[t_A], task_bases[t_B]
        X_B_centered = task_acts_centered[t_B]

        overlap = compute_normalized_subspace_overlap(V_A, V_B)
        unexplained = compute_unexplained_variance_fraction(X_B_centered, V_A)

        pair_overlaps.append(overlap)
        pair_unexplained.append(unexplained)

    layer_mean_overlap = np.mean(pair_overlaps)
    layer_mean_unexplained = np.mean(pair_unexplained)

    mean_overlaps_per_layer.append(layer_mean_overlap)
    mean_unexplained_per_layer.append(layer_mean_unexplained)

    print(
        f"--> Layer {l_idx + 1:2d}/{num_layers:2d} | Subspace Overlap: {layer_mean_overlap:.4f} | GPM Unexplained Energy: {layer_mean_unexplained:.4f}"
    )


# --- 3. PLOTTING THE TRAJECTORY ACROSS DEPTH ---
layers_x = np.arange(1, num_layers + 1)

fig, ax1 = plt.subplots(figsize=(10, 6), dpi=100)

color_overlap = "dodgerblue"
ax1.plot(
    layers_x,
    mean_overlaps_per_layer,
    marker="o",
    color=color_overlap,
    linewidth=2.5,
    markersize=7,
    label="Normalized Subspace Overlap Score",
)
ax1.set_xlabel("Layer Index (Depth)", fontsize=11, weight="bold")
ax1.set_ylabel(
    "Mean Subspace Overlap (190 Pairs)",
    color=color_overlap,
    fontsize=11,
    weight="bold",
)
ax1.set_ylim(0, 1.0)
ax1.grid(True, linestyle=":", alpha=0.6)

ax2 = ax1.twinx()
color_unexplained = "crimson"
ax2.plot(
    layers_x,
    mean_unexplained_per_layer,
    marker="s",
    color=color_unexplained,
    linewidth=2.5,
    markersize=7,
    linestyle="--",
    label="GPM Unexplained Energy Fraction",
)
ax2.set_ylabel(
    "Mean Unexplained Energy Fraction",
    color=color_unexplained,
    fontsize=11,
    weight="bold",
)
ax2.set_ylim(0, 1.0)

plt.title(
    f"Cross-Task Subspace Alignment Profile Across Depth ({num_tasks} Tasks, 190 Pairs)\nEvaluating SVD Subspace Coherence & GPM Expansion Need",
    fontsize=12,
    weight="bold",
)
plt.xticks(layers_x)

# Combine legends
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch

# --- 1. COMPUTATIONAL ENGINES ---


def compute_all_pairs_energy_subspace_overlap(
    task_pre_acts, layer_idx, energy_fractions, act_fn=torch.tanh
):
    """Computes pairwise subspace overlap by sweeping target variance energy fractions (0.0 to 1.0)."""
    num_tasks = len(task_pre_acts)
    task_singular_vectors = []
    task_cumsum_energy = []

    for t in range(num_tasks):
        layer_pre = task_pre_acts[t][layer_idx]
        act = (
            act_fn(layer_pre).detach().cpu().numpy()
            if isinstance(layer_pre, torch.Tensor)
            else act_fn(torch.tensor(layer_pre)).numpy()
        )

        act_centered = act - np.mean(act, axis=0, keepdims=True)
        _, S, Vt = np.linalg.svd(act_centered, full_matrices=False)

        singular_energy = S**2
        total_energy = np.sum(singular_energy) + 1e-12
        cumsum_E = np.cumsum(singular_energy) / total_energy

        task_singular_vectors.append(Vt.T)
        task_cumsum_energy.append(cumsum_E)

    pair_indices = list(itertools.combinations(range(num_tasks), 2))
    num_pairs = len(pair_indices)
    num_thresholds = len(energy_fractions)

    all_overlap_curves = np.zeros((num_pairs, num_thresholds))
    all_k_curves = np.zeros((num_pairs, num_thresholds))

    for p_idx, (i, j) in enumerate(pair_indices):
        V_full_A, cumsum_A = task_singular_vectors[i], task_cumsum_energy[i]
        V_full_B, cumsum_B = task_singular_vectors[j], task_cumsum_energy[j]

        for e_idx, target_E in enumerate(energy_fractions):
            k_A = min(
                np.searchsorted(cumsum_A, target_E) + 1, V_full_A.shape[1]
            )
            k_B = min(
                np.searchsorted(cumsum_B, target_E) + 1, V_full_B.shape[1]
            )

            V_A, V_B = V_full_A[:, :k_A], V_full_B[:, :k_B]

            proj_matrix = V_A.T @ V_B
            frobenius_sq = np.linalg.norm(proj_matrix, ord="fro") ** 2
            overlap = frobenius_sq / min(k_A, k_B)

            all_overlap_curves[p_idx, e_idx] = overlap
            all_k_curves[p_idx, e_idx] = (k_A + k_B) / 2.0

    return all_overlap_curves, all_k_curves


def compute_all_pairs_k_subspace_overlap(
    task_pre_acts, layer_idx, max_k=100, act_fn=torch.tanh
):
    """Computes pairwise subspace overlap by sweeping discrete singular vector counts (k = 1 to max_k)."""
    num_tasks = len(task_pre_acts)
    task_singular_vectors = []
    task_cumsum_energy = []

    for t in range(num_tasks):
        layer_pre = task_pre_acts[t][layer_idx]
        act = (
            act_fn(layer_pre).detach().cpu().numpy()
            if isinstance(layer_pre, torch.Tensor)
            else act_fn(torch.tensor(layer_pre)).numpy()
        )

        act_centered = act - np.mean(act, axis=0, keepdims=True)
        _, S, Vt = np.linalg.svd(act_centered, full_matrices=False)

        singular_energy = S**2
        total_energy = np.sum(singular_energy) + 1e-12
        cumsum_E = np.cumsum(singular_energy) / total_energy

        task_singular_vectors.append(Vt.T)
        task_cumsum_energy.append(cumsum_E)

    pair_indices = list(itertools.combinations(range(num_tasks), 2))
    k_range = np.arange(1, max_k + 1)

    all_overlap_curves = np.zeros((len(pair_indices), len(k_range)))
    all_var_curves = np.zeros((len(pair_indices), len(k_range)))

    for p_idx, (i, j) in enumerate(pair_indices):
        V_full_A, cumsum_A = task_singular_vectors[i], task_cumsum_energy[i]
        V_full_B, cumsum_B = task_singular_vectors[j], task_cumsum_energy[j]

        for idx, k in enumerate(k_range):
            k_A = min(k, V_full_A.shape[1])
            k_B = min(k, V_full_B.shape[1])

            V_A, V_B = V_full_A[:, :k_A], V_full_B[:, :k_B]

            proj_matrix = V_A.T @ V_B
            frobenius_sq = np.linalg.norm(proj_matrix, ord="fro") ** 2
            overlap = frobenius_sq / min(k_A, k_B)

            all_overlap_curves[p_idx, idx] = overlap
            all_var_curves[p_idx, idx] = (
                cumsum_A[k_A - 1] + cumsum_B[k_B - 1]
            ) / 2.0

    return k_range, all_overlap_curves, all_var_curves


def run_bootstrap_ci(data_matrix, num_bootstraps=1000, ci_level=95):
    """Computes mean curve and percentile-based bootstrap confidence intervals."""
    num_pairs, num_points = data_matrix.shape
    boot_means = np.zeros((num_bootstraps, num_points))

    rng = np.random.default_rng(seed=42)
    for b in range(num_bootstraps):
        boot_indices = rng.choice(num_pairs, size=num_pairs, replace=True)
        boot_means[b, :] = np.mean(data_matrix[boot_indices, :], axis=0)

    lower_p = (100 - ci_level) / 2.0
    upper_p = 100 - lower_p

    mean_curve = np.mean(data_matrix, axis=0)
    ci_lower = np.percentile(boot_means, lower_p, axis=0)
    ci_upper = np.percentile(boot_means, upper_p, axis=0)

    return mean_curve, ci_lower, ci_upper


# --- 2. CONFIGURATION & RUNTIME ---

TARGET_LAYERS = [1, 4, 7]  # 0-indexed layers to compare
MAX_K_SWEEP = 120  # Max component count for Right Plot
ACTIVATION_FN = torch.tanh

energy_fractions = np.linspace(0.02, 1.0, 99)
x_axis_energy = energy_fractions * 100

target_layers_list = (
    [TARGET_LAYERS] if isinstance(TARGET_LAYERS, int) else TARGET_LAYERS
)
colors = plt.cm.plasma(np.linspace(0.1, 0.85, len(target_layers_list)))

# Initialize Dual Subplots Figure
fig, (ax1_left, ax2_left) = plt.subplots(1, 2, figsize=(20, 7))
ax1_right = ax1_left.twinx()
ax2_right = ax2_left.twinx()

print(f"Beginning Dual Subspace Analysis across Layers: {target_layers_list}")

for idx, l_idx in enumerate(target_layers_list):
    color = colors[idx]

    # --- LEFT PANEL COMPUTATION (Variance Sweep) ---
    e_overlap_mat, e_k_mat = compute_all_pairs_energy_subspace_overlap(
        task_pre_acts, l_idx, energy_fractions, act_fn=ACTIVATION_FN
    )
    e_o_mean, e_o_low, e_o_high = run_bootstrap_ci(e_overlap_mat)
    e_k_mean, e_k_low, e_k_high = run_bootstrap_ci(e_k_mat)

    # Left Panel Plotting
    line1 = ax1_left.plot(
        x_axis_energy,
        e_o_mean,
        color=color,
        linewidth=2.2,
        label=f"Layer {l_idx + 1} Overlap",
    )
    ax1_left.fill_between(
        x_axis_energy, e_o_low, e_o_high, color=color, alpha=0.12
    )

    line2 = ax1_right.plot(
        x_axis_energy,
        e_k_mean,
        color=color,
        linestyle="--",
        linewidth=1.6,
        alpha=0.85,
        label=f"Layer {l_idx + 1} Active k",
    )
    ax1_right.fill_between(
        x_axis_energy, e_k_low, e_k_high, color=color, alpha=0.05
    )

    # --- RIGHT PANEL COMPUTATION (Fixed Vector k Sweep) ---
    k_range, k_overlap_mat, k_var_mat = compute_all_pairs_k_subspace_overlap(
        task_pre_acts, l_idx, max_k=MAX_K_SWEEP, act_fn=ACTIVATION_FN
    )
    k_o_mean, k_o_low, k_o_high = run_bootstrap_ci(k_overlap_mat)
    k_v_mean, k_v_low, k_v_high = run_bootstrap_ci(k_var_mat)

    # Right Panel Plotting
    ax2_left.plot(
        k_range,
        k_o_mean,
        color=color,
        linewidth=2.2,
        label=f"Layer {l_idx + 1} Overlap",
    )
    ax2_left.fill_between(k_range, k_o_low, k_o_high, color=color, alpha=0.12)

    ax2_right.plot(
        k_range,
        k_v_mean * 100,
        color=color,
        linestyle="--",
        linewidth=1.6,
        alpha=0.85,
        label=f"Layer {l_idx + 1} Var %",
    )
    ax2_right.fill_between(
        k_range, k_v_low * 100, k_v_high * 100, color=color, alpha=0.05
    )


# --- 3. FORMATTING LEFT SUBPLOT (Energy Mass % x-Axis) ---
ax1_left.set_xlabel(
    "Cumulative Task SVD Variance Mass (%)", fontsize=11, fontweight="bold"
)
ax1_left.set_ylabel(
    "Normalized Subspace Overlap Score", fontsize=11, fontweight="bold"
)
ax1_left.set_ylim(0.0, 1.0)
ax1_left.set_xlim(0, 100)
ax1_left.grid(True, linestyle=":", alpha=0.6)

ax1_right.set_ylabel(
    "Mean Singular Vectors Required (k)",
    fontsize=11,
    fontweight="bold",
    color="dimgray",
)

ax1_left.set_title(
    "A. Alignment vs. Cumulative Variance Mass (%)\n(Energy-Normalized Comparison)",
    fontsize=12,
    fontweight="bold",
    pad=10,
)

# Legends for Left Subplot
lines_left = ax1_left.get_lines() + ax1_right.get_lines()
labels_left = [l.get_label() for l in lines_left]
ax1_left.legend(lines_left, labels_left, loc="upper left", frameon=True)


# --- 4. FORMATTING RIGHT SUBPLOT (Vector Count k x-Axis) ---
ax2_left.set_xlabel(
    "Top Principal Components Retained (k)", fontsize=11, fontweight="bold"
)
ax2_left.set_ylabel(
    "Normalized Subspace Overlap Score", fontsize=11, fontweight="bold"
)
ax2_left.set_ylim(0.0, 1.0)
ax2_left.set_xlim(1, MAX_K_SWEEP)
ax2_left.grid(True, linestyle=":", alpha=0.6)

ax2_right.set_ylabel(
    "Mean Cumulative Variance Explained (%)",
    fontsize=11,
    fontweight="bold",
    color="dimgray",
)
ax2_right.set_ylim(0, 100)

ax2_title_str = ", ".join([str(l + 1) for l in target_layers_list])
ax2_left.set_title(
    "B. Alignment vs. Fixed Subspace Dimension (k)\n(Capacity-Normalized Comparison)",
    fontsize=12,
    fontweight="bold",
    pad=10,
)

# Legends for Right Subplot
lines_right = ax2_left.get_lines() + ax2_right.get_lines()
labels_right = [l.get_label() for l in lines_right]
ax2_left.legend(lines_right, labels_right, loc="lower right", frameon=True)


plt.suptitle(
    f"Dual Perspective: Subspace Overlap and Spectral Density at Layer(s): {ax2_title_str}",
    fontsize=14,
    fontweight="bold",
    y=0.99,
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

# --- 1. CONFIGURATION FOR 2x2 COMPARISON ---

# Select task index to analyze (0-indexed, e.g., 0 for Task 1)
TARGET_TASK_IDX = 0

# Select two layers to compare (0-indexed, e.g., 0 for Layer 1, 4 for Layer 5)
LAYER_IDX_A = 0  # Left Column (e.g., Layer 1)
LAYER_IDX_B = 4  # Right Column (e.g., Layer 5)

ACTIVATION_TYPE = "tanh"  # Options: 'tanh', 'relu'


# --- 2. HELPER FUNCTIONS ---


def get_linear_weights(model):
    """Extracts weight matrices from all linear layers in the model."""
    linear_weights = []
    for module in model.modules():
        if isinstance(module, nn.Linear):
            linear_weights.append(module.weight.detach().cpu().numpy())
    return linear_weights


def compute_activation_derivative(pre_acts, act_type="tanh"):
    """Computes element-wise derivative of activation function given pre-activations."""
    if isinstance(pre_acts, torch.Tensor):
        pre_acts = pre_acts.detach().cpu().numpy()

    if act_type.lower() == "tanh":
        # d/dz tanh(z) = 1 - tanh^2(z)
        return 1.0 - np.tanh(pre_acts) ** 2
    elif act_type.lower() == "relu":
        # d/dz relu(z) = H(z) (Heaviside step)
        return (pre_acts > 0).astype(np.float32)
    else:
        raise ValueError(f"Unsupported activation function: {act_type}")


def compute_layer_jacobian_eigenvalues(
    model, task_pre_acts, task_idx, layer_idx, act_type="tanh"
):
    """Computes the effective layerwise Jacobian J = D @ W and returns its complex eigenvalues.

    J_l = diag(mean_batch(sigma'(z_l))) @ W_l
    """
    weights = get_linear_weights(model)
    W_l = weights[layer_idx]  # Shape: [out_features, in_features]

    # Retrieve pre-activations for chosen task and layer: shape [BATCH_SIZE, hidden_size]
    z_l = task_pre_acts[task_idx][layer_idx]

    # 1. Compute derivative sigma'(z_l)
    sigma_prime = compute_activation_derivative(z_l, act_type=act_type)

    # 2. Average derivative gate over batch dimension -> shape: [hidden_size]
    d_l = np.mean(sigma_prime, axis=0)

    # 3. Construct effective Jacobian operator: J_l = diag(d_l) @ W_l
    J_l = np.diag(d_l) @ W_l

    # 4. Compute full complex eigenvalue spectrum
    eigenvalues = np.linalg.eigvals(J_l)

    return eigenvalues


# --- 3. COMPUTATION & SPECTRUM EXTRACTION FOR BOTH RUNS ---

print(f"Computing 2x2 Jacobian spectra for Task {TARGET_TASK_IDX + 1}...")

# Row 1: Gaussian Initialisation
eigs_gauss_A = compute_layer_jacobian_eigenvalues(
    runs_data["gaussian"]["model"],
    runs_data["gaussian"]["task_pre_acts"],
    TARGET_TASK_IDX,
    LAYER_IDX_A,
    act_type=ACTIVATION_TYPE,
)
eigs_gauss_B = compute_layer_jacobian_eigenvalues(
    runs_data["gaussian"]["model"],
    runs_data["gaussian"]["task_pre_acts"],
    TARGET_TASK_IDX,
    LAYER_IDX_B,
    act_type=ACTIVATION_TYPE,
)

# Row 2: Heavy-Tailed Initialisation
eigs_ht_A = compute_layer_jacobian_eigenvalues(
    runs_data["heavy_tailed"]["model"],
    runs_data["heavy_tailed"]["task_pre_acts"],
    TARGET_TASK_IDX,
    LAYER_IDX_A,
    act_type=ACTIVATION_TYPE,
)
eigs_ht_B = compute_layer_jacobian_eigenvalues(
    runs_data["heavy_tailed"]["model"],
    runs_data["heavy_tailed"]["task_pre_acts"],
    TARGET_TASK_IDX,
    LAYER_IDX_B,
    act_type=ACTIVATION_TYPE,
)

# Global axis limit synchronization across ALL 4 panels
all_eigs = [eigs_gauss_A, eigs_gauss_B, eigs_ht_A, eigs_ht_B]
max_global_radius = max(np.max(np.abs(e)) for e in all_eigs)
axis_limit = max(max_global_radius, 1.2) * 1.1  # Ensures unit circle fits cleanly


# --- 4. 2x2 GRID COMPLEX PLANE PLOTTING ---

fig, axes = plt.subplots(2, 2, figsize=(14, 13))

# Unit circle parametrization
theta = np.linspace(0, 2 * np.pi, 300)
unit_circle_x = np.cos(theta)
unit_circle_y = np.sin(theta)

# Define 2x2 grid panel specifications
grid_specs = [
    # (Row, Col, Eigenvalues, Layer Index, Label, Accent Color)
    (
        0,
        0,
        eigs_gauss_A,
        LAYER_IDX_A,
        f"Gaussian Init — Layer {LAYER_IDX_A + 1}",
        "teal",
    ),
    (
        0,
        1,
        eigs_gauss_B,
        LAYER_IDX_B,
        f"Gaussian Init — Layer {LAYER_IDX_B + 1}",
        "darkcyan",
    ),
    (
        1,
        0,
        eigs_ht_A,
        LAYER_IDX_A,
        f"Heavy-Tailed Init — Layer {LAYER_IDX_A + 1}",
        "indigo",
    ),
    (
        1,
        1,
        eigs_ht_B,
        LAYER_IDX_B,
        f"Heavy-Tailed Init — Layer {LAYER_IDX_B + 1}",
        "crimson",
    ),
]

for row, col, eigs, l_idx, title_prefix, color in grid_specs:
    ax = axes[row, col]

    # 1. Reference grid & origin lines
    ax.axhline(0, color="gray", linestyle=":", alpha=0.6, linewidth=1.0)
    ax.axvline(0, color="gray", linestyle=":", alpha=0.6, linewidth=1.0)

    # 2. Unit circle boundary (|lambda| = 1)
    ax.plot(
        unit_circle_x,
        unit_circle_y,
        color="black",
        linestyle="--",
        linewidth=1.8,
        label=r"Unit Boundary ($|\lambda| = 1$)",
        zorder=2,
    )

    # 3. Scatter plot complex eigenvalues
    ax.scatter(
        eigs.real,
        eigs.imag,
        color=color,
        alpha=0.65,
        s=26,
        edgecolors="none",
        label=f"Eigenvalues (N={len(eigs)})",
        zorder=3,
    )

    # 4. Diagnostics: Spectral radius and zero mass density
    zero_count = np.sum(np.abs(eigs) < 0.05)
    zero_ratio = (zero_count / len(eigs)) * 100
    max_r = np.max(np.abs(eigs))

    # Formatting & Limits
    ax.set_aspect("equal", "box")
    ax.set_xlim(-axis_limit, axis_limit)
    ax.set_ylim(-axis_limit, axis_limit)

    ax.set_xlabel(r"Real Part $\operatorname{Re}(\lambda)$", fontweight="bold")
    ax.set_ylabel(
        r"Imaginary Part $\operatorname{Im}(\lambda)$", fontweight="bold"
    )

    ax.set_title(
        f"{title_prefix}",
        # f"Max Radius $\\rho(J) = {max_r:.2f}$ | Zero-Mass ($|\\lambda|<0.05$): {zero_ratio:.1f}%",
        fontsize=11,
        fontweight="bold",
        pad=10,
    )
    ax.grid(True, linestyle=":", alpha=0.4)
    ax.legend(loc="upper right", frameon=True, fontsize=8.5)

plt.suptitle(
    f"Layerwise Non-Hermitian Jacobian Spectra Comparison (Task {TARGET_TASK_IDX + 1})\n"
    f"Gaussian (Row 1) vs. Heavy-Tailed (Row 2) Across Layer Depth",
    fontsize=14,
    fontweight="bold",
    y=0.99,
)

plt.tight_layout()
plt.show()